# application_train - exploration

Looking at the data. Nothing here ships; what survives goes into
`src/credit_engine/features/` with tests.

Pass 1 (sec 1-4): what predicts default. Done.
Pass 2 (sec 5-9): what each column needs done to it before a model sees it.

Done when every column has a bucket in sec 9.

## setup

In [ ]:
# Will explore the data here!
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. target

`TARGET = 1` means payment difficulties.

In [ ]:
df = pd.read_csv("../data/raw/application_train.csv")
df.head(10)

In [ ]:
df["TARGET"].value_counts()
# 1 -> have payments difficulties
# 0 -> have no payment difficulties

8.07% positive. Approve-everyone = 92% accurate and useless, so no accuracy anywhere.

## 2. missingness

Not all the same thing. Some random, some in blocks, some tied to the target.
Need to tell them apart before imputing.

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss

Building columns (COMMONAREA, LIVINGAPARTMENTS, ...) sit ~50-70% missing. Check if they move together.

In [ ]:
miss[miss.index.str.startswith("COMMONAREA")]

In [ ]:
cols = [c for c in df.columns if c.startswith("COMMONAREA")]
df[cols].corr()

They do. One block, missing together => one indicator for the block, not 50.
Looks MAR (tracks housing type, not target).

## 3. correlation with target

Linear only, misses interactions. Just to see which families matter.

In [ ]:
num = df.select_dtypes(include="number").drop(columns=["SK_ID_CURR", "TARGET"])
num.corrwith(df["TARGET"]).sort_values()

## 4. EXT_SOURCE

Three anonymous third-party scores. Top of the correlation scan.

In [ ]:
EXT_SOURCE_MISSING = (
    df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].isna().mean().sort_values(ascending=False)
)

In [ ]:
EXT_SOURCE_MISSING

In [ ]:
EXT_SOURCE_MISSING.describe()
EXT_SOURCE_MISSING.plot(kind="bar", title="Missing values in EXT_SOURCE columns")

EXT_SOURCE_1 56% missing, _3 20%, _2 almost none. Different columns despite the name.

Does the score separate risk, and does missing mean anything?

In [ ]:
df.groupby(pd.qcut(df["EXT_SOURCE_2"], 10), observed=True)["TARGET"].agg(["mean", "count"])

In [ ]:
df.groupby(df["EXT_SOURCE_1"].isna())["TARGET"].agg(["mean", "count"])

EXT_SOURCE_2: 18.4% default bottom decile -> 3.0% top, monotonic. Strongest single feature.

EXT_SOURCE_1: missing rows default at a different rate. MNAR, so keep an indicator.
Imputing it away loses signal.

Keep it, label it honestly in the API, flag non-actionable (ADR #14).
Cost of dropping it gets measured in Phase 2.

---

# pass 2 - preprocessing

Above was which features matter. Below is what to do to each column.

## 5. DAYS_EMPLOYED sentinel

No missing values, still broken.

In [ ]:
df["DAYS_EMPLOYED"].describe()

max = 365243 days = ~1000 years. It's a NULL written as a number, so `.isna()` misses it.

In [ ]:
SENTINEL = 365243

sent = df["DAYS_EMPLOYED"] == SENTINEL
print(f"sentinel rows: {sent.sum():,} ({sent.mean():.2%})")

# who are they, and do they default differently?
print()
print(df.loc[sent, "NAME_INCOME_TYPE"].value_counts().head())
print()
print(df.groupby(sent)["TARGET"].agg(["mean", "count"]))

18.01% of rows, almost all pensioners. Means "not employed", not "unknown".
Default 5.40% vs 8.66% employed, so the flag is predictive.

=> `sentinel-fix`: 365243 -> NaN + `DAYS_EMPLOYED_SENTINEL` indicator.
Leaving it puts a 1000-year tenure in every coefficient. Dropping the rows throws
away a segment that defaults at half the base rate.

Check the other DAYS_ columns for the same thing.

In [ ]:
days_cols = [c for c in df.columns if c.startswith("DAYS_")]
df[days_cols].agg(["min", "max", "mean"]).T.assign(has_sentinel=lambda t: t["max"] == SENTINEL)

Only DAYS_EMPLOYED. The other four are all <=0 with no sentinel, so the fix is
one column, not a family rule.

## 6. categoricals

Haven't looked at these at all. Can't pick an encoder without cardinality.

In [ ]:
obj = df.select_dtypes("object")
print(f"{obj.shape[1]} categorical columns\n")

pd.DataFrame(
    {
        "levels": obj.nunique(),
        "missing_%": (obj.isna().mean() * 100).round(2),
    }
).sort_values("levels", ascending=False)

16 object columns.

- ORGANIZATION_TYPE 58 levels, OCCUPATION_TYPE 18, other 14 are <=8.
  One-hot on everything adds ~140 cols, most from those two. Decide them separately.
- 4 cols >47% missing (FONDKAPREMONT 68%, WALLSMATERIAL 51%, HOUSETYPE 50%,
  EMERGENCYSTATE 47%) - same building block as sec 2. For a categorical, missing can
  just be its own level.
- OCCUPATION_TYPE 31% missing. Different reason? check below.

In [ ]:
# is missing OCCUPATION_TYPE explained by not being employed?
occ_missing = df["OCCUPATION_TYPE"].isna()
print(df.loc[occ_missing, "NAME_INCOME_TYPE"].value_counts().head())
print()
print(df.groupby(occ_missing)["TARGET"].agg(["mean", "count"]))

Missing OCCUPATION_TYPE defaults at 6.5% vs 8.8%. Informative, so not benign.

But it's mostly not a separate fact: 55,372 of the 96,391 missing rows are the
same pensioners as the DAYS_EMPLOYED sentinel. "No occupation" and "not employed"
are one thing recorded twice.

=> encode with MISSING as its own level. No separate indicator, DAYS_EMPLOYED_SENTINEL
already carries it.

CODE_GENDER has 3 values, not 2.

In [ ]:
df["CODE_GENDER"].value_counts(dropna=False)

XNA, 4 rows. All TARGET=0, nothing else unusual.

=> map XNA -> NaN, leaves F/M. Keeping it makes a one-hot column that is 1 for four
rows out of 307k: no signal, and it would show up in SHAP output on a protected
attribute. Can't drop the rows - serving has to handle whatever arrives.

## 7. binary flags

32 numeric cols are 0/1 only. Nothing to encode, but a flag that's almost always
0 costs compute and explains nothing.

In [ ]:
num = df.select_dtypes("number").drop(columns=["SK_ID_CURR", "TARGET"])
binary = [c for c in num.columns if num[c].dropna().isin([0, 1]).all()]

pd.DataFrame(
    {
        "positive_rate_%": (num[binary].mean() * 100).round(3),
        "corr_with_target": num[binary].corrwith(df["TARGET"]).round(4),
    }
).sort_values("positive_rate_%")

FLAG_DOCUMENT_3 is 71% positive, _6 and _8 ~8%, everything else under 2%.
The tail goes down to FLAG_DOCUMENT_12 at 2 positive rows.

Cutoff rule: keep a flag only if its positive group would contain at least 10
expected defaults. At the 8.07% base rate that's **123 positive rows**.

Picked that way because the flag is only useful if you can estimate a default rate
inside it; under ~10 events the estimate is noise. Not a round number chosen to
look tidy - it falls out of the base rate.

Drops 7: _2, _4, _7, _10, _12, _17, _21. Keeps 13.

## 8. near-constant / duplicate

Last structural sweep.

In [ ]:
nunique = df.nunique()

print("single-valued columns:")
print(nunique[nunique <= 1].to_string() or "  none")

print("\ntop-value dominance > 99% (near-constant):")
dominance = df.apply(lambda s: s.value_counts(normalize=True, dropna=False).iloc[0])
print(
    (dominance[dominance > 0.99].sort_values(ascending=False) * 100).round(2).to_string()
    or "  none"
)

In [ ]:
# exact duplicates - compare full columns, not a sample
seen, dups = {}, []
for c in df.columns.drop("SK_ID_CURR"):
    k = pd.util.hash_pandas_object(df[c].astype(str), index=False).sum()
    if k in seen and df[c].equals(df[seen[k]]):
        dups.append((seen[k], c))
    else:
        seen.setdefault(k, c)
print("duplicate columns:", dups or "none")

No single-valued columns, no exact duplicates.

18 columns are >99% one value, but 16 of those are document flags already handled
by the sec 7 rule. Two left:

- FLAG_MOBIL: 1 in 307,510 rows. Constant in practice => drop.
- FLAG_CONT_MOBILE: 574 zeros, above the 123 cutoff => keep.

(First pass at duplicates hashed only the first 5000 rows and reported the rare
document flags as duplicates. They're all-zero that far in. Compare full columns.)

## 9. column buckets

Every column gets one bucket + a reason.

| bucket | meaning |
|---|---|
| `drop` | constant, duplicate, or an id |
| `passthrough` | numeric, fine as-is |
| `impute` | missing, absence means nothing |
| `impute+indicator` | missing, and the absence is itself signal (EXT_SOURCE_1) |
| `encode` | categorical |
| `sentinel-fix` | placeholder pretending to be a value (DAYS_EMPLOYED) |

Assigned by rule per family, not typed out 122 times - the rules are what gets
defended, and they re-run if the data changes.

In [ ]:
import re

BASE = df["TARGET"].mean()
MIN_DEF = 10  # min expected defaults to estimate a rate
MIN_POS = int(MIN_DEF / BASE)  # -> min positive rows a binary flag needs
print(f"base rate {BASE:.4f} -> flag cutoff {MIN_POS} positive rows")

bucket, reason = {}, {}


def put(col, b, r):
    bucket[col], reason[col] = b, r


put("SK_ID_CURR", "drop", "identifier, not a feature")
put("TARGET", "target", "the label")
put("FLAG_MOBIL", "drop", "constant: 1 in 307,510 rows")

# document flags: keep only those with enough positives to estimate a rate
for c in [c for c in df.columns if c.startswith("FLAG_DOCUMENT_")]:
    pos = int(df[c].sum())
    if pos < MIN_POS:
        put(c, "drop", f"{pos} positive rows < {MIN_POS}")
    else:
        put(c, "passthrough", f"binary, {pos:,} positives")

# building block: 19 stems x AVG/MODE/MEDI, AVG~MEDI r>0.99
for c in [c for c in df.columns if re.search(r"_(AVG|MODE|MEDI)$", c)]:
    if df[c].dtype == object:
        put(c, "encode", f"categorical, {df[c].isna().mean():.0%} missing -> MISSING level")
    elif c.endswith("_AVG") or re.sub(r"_(MODE|MEDI)$", "_AVG", c) not in df.columns:
        put(c, "impute+indicator", "building block, one shared missing indicator")
    else:
        put(c, "drop", "redundant suffix, r>0.99 with _AVG")

put("EXT_SOURCE_1", "impute+indicator", "MNAR, absence predicts target")
put("EXT_SOURCE_2", "impute", "0.2% missing, no signal in absence")
put("EXT_SOURCE_3", "impute+indicator", "20% missing, same family as _1")

put("DAYS_EMPLOYED", "sentinel-fix", "365243 = not employed (18.0%); NaN + indicator")
for c in ["DAYS_BIRTH", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE"]:
    put(c, "passthrough", "no sentinel, no missing")

put("OWN_CAR_AGE", "impute+indicator", "66% missing = no car, mirrors FLAG_OWN_CAR")
put("OCCUPATION_TYPE", "encode", "31% missing, 6.5% vs 8.8% default -> MISSING is a level")
put("CODE_GENDER", "encode", "XNA (4 rows) -> NaN; leaves F/M")
put("ORGANIZATION_TYPE", "encode", "58 levels, needs grouping before one-hot")
put("WEEKDAY_APPR_PROCESS_START", "encode", "7 levels")
put("HOUR_APPR_PROCESS_START", "passthrough", "0-23 int, ordinal enough")

for c in [c for c in df.columns if c.startswith("AMT_REQ_CREDIT_BUREAU_")]:
    put(c, "impute+indicator", "13.5% missing = no bureau record (thin file)")

for c in [c for c in df.columns if c.startswith(("OBS_", "DEF_"))]:
    put(c, "impute", "0.3% missing; window undocumented (eval.md suspect)")

# whatever is left
for c in df.columns:
    if c in bucket:
        continue
    miss = df[c].isna().mean()
    if df[c].dtype == object:
        put(c, "encode", f"{df[c].nunique()} levels")
    elif miss > 0:
        put(c, "impute", f"{miss:.1%} missing, benign")
    else:
        put(c, "passthrough", "numeric, complete")

spec = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing_%": (df.isna().mean() * 100).round(2),
        "levels": df.nunique(),
        "bucket": pd.Series(bucket),
        "reason": pd.Series(reason),
    }
)
spec.groupby("bucket").size().sort_values(ascending=False)

37 drop, 35 passthrough, 24 impute+indicator, 16 encode, 8 impute, 1 sentinel-fix.

The 37 drops are mostly one decision: 28 of them are the _MODE/_MEDI copies of the
building block. 7 are rare document flags, plus FLAG_MOBIL and SK_ID_CURR.

84 features go into the baseline before any encoding.

In [ ]:
assert spec["bucket"].notna().all(), "unassigned columns remain"
spec.to_csv("../docs/column_spec.csv")
print(f"wrote docs/column_spec.csv - {len(spec)} columns")
spec[spec.bucket == "impute+indicator"]

## then the code moves

1. Table above is the spec.
2. Write the transform in `src/credit_engine/features/` with its test, one bucket at a time.
3. Import it back here and check the output against the raw column.

Not: build the whole transform in cells and copy it over later. That arrives untested
and depends on cell order.

Open, for Phase 2 not now:
- ORGANIZATION_TYPE 58 levels needs a grouping rule before one-hot.
- OBS_/DEF_ social circle kept but the observation window is undocumented (eval.md).